In [ ]:
import os
import pickle
import sys

root_dir = os.path.abspath("..")
data_dir = os.path.join(root_dir, "data")
os.makedirs(data_dir, exist_ok=True)
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux
from current_setpoints.models.forward_model import ForwardModel, Fault
from current_setpoints.optimization.optimizer import StaticOptimizer
from current_setpoints.optimization.grid import calculate_grid
from current_setpoints.optimization.data import grid_to_data
from current_setpoints.utils.plotting import plot_grid_segments

In [ ]:
CURR_MAX = 30.0
VOLT_MAX = 13.0
OMEGA_MAX = 1800.0  # mechanical RPM

SLSQP_OPTS = {"disp": False, "ftol": 1e-8, "maxiter": 500}
# ~3-4s/cell with the neural flux model (SLSQP needs many network evaluations
# per solve) - 15x15 is ~15 min; raise toward 101x101 (the old ConstantFlux
# resolution) only if you can let it run for hours.
GRID_OPTS = {"n_torq": 15, "n_omega": 15, "torq_min": 0.0, "omega_min": 0.0}

In [ ]:
drive = ieee_machine2_trained_neural_flux(curr_max=CURR_MAX, volt_max=VOLT_MAX, omega_max=OMEGA_MAX)
fwd = ForwardModel(drive, n_theta=700, fault=Fault(()))
optimizer = StaticOptimizer(fwd, opts=SLSQP_OPTS)

In [ ]:
pkl_path = os.path.join(data_dir, "baseline_map_neural_flux.pkl")

if os.path.isfile(pkl_path):
    with open(pkl_path, "rb") as f:
        grid = pickle.load(f)
else:
    grid = calculate_grid(optimizer=optimizer, fwd=fwd, opts=GRID_OPTS, mode="standard")
    with open(pkl_path, "wb") as f:
        pickle.dump(grid, f)

In [ ]:
data = grid_to_data(grid, k_skip=1)
plot_grid_segments(data, title="Machine2 baseline map (neural flux + L(i))")